# PPOModalCuriosity Tutorial

This notebook documents the first-pass `ppo_modal_curiosity` trainer plugin inside `ml-agents-psych`.

Its goal is narrow:

- keep stock PPO semantics as intact as possible
- avoid patching base `mlagents` in the first pass
- keep one outer `curiosity` reward signal
- compute that curiosity reward from weighted modality-specific curiosity branches


## What This Package Brings

`ml-agents-psych` now contains two plugin surfaces:

- a `StatsWriter` plugin for psychometric transition logging
- a `trainer_type` plugin for `ppo_modal_curiosity`

That split is intentional. Logging extensions and trainer extensions live in one project-owned package, but they stay operationally separate and can be enabled independently through normal ML-Agents plugin registration.


## Why A Trainer Shim Instead Of A Base Patch

The active deployment lane is built around released `mlagents==1.1.0` plus a standalone plugin package, not around a copied local trainer fork.

Project path references:

- `Docker/CustomRIS_audiov0.1/source_of_truth.yaml`
- `Docker/CustomRIS_audiov0.1/README.md`

ML-Agents exposes `mlagents.trainer_type` as the official plugin seam for custom trainers. That makes a thin trainer shim the cleanest first-pass injection point for modal curiosity.


## Upstream Classes This Builds On

These are the main upstream classes and file paths the plugin architecture builds on:

- trainer plugin registration: `Docker/CustomRIS_40X/ml-agents-local/ml-agents/mlagents/plugins/trainer_type.py`
- stock PPO trainer: `Docker/CustomRIS_40X/ml-agents-local/ml-agents/mlagents/trainers/ppo/trainer.py`
- stock PPO optimizer and settings: `Docker/CustomRIS_40X/ml-agents-local/ml-agents/mlagents/trainers/ppo/optimizer_torch.py`
- reward-provider construction: `Docker/CustomRIS_40X/ml-agents-local/ml-agents/mlagents/trainers/optimizer/torch_optimizer.py`
- stock curiosity reward provider: `Docker/CustomRIS_40X/ml-agents-local/ml-agents/mlagents/trainers/torch_entities/components/reward_providers/curiosity_reward_provider.py`

Inside this package, the implementation lives at:

- `mlagents_psych/modal_curiosity/config.py`
- `mlagents_psych/modal_curiosity/selection.py`
- `mlagents_psych/modal_curiosity/provider.py`
- `mlagents_psych/modal_curiosity/optimizer.py`
- `mlagents_psych/modal_curiosity/trainer.py`


## Stock Curiosity Math

Stock ICM uses one shared encoder over the full observation set:

$$z_t = \phi(o_t)$$

$$\hat{z}_{t+1} = f(z_t, u_t)$$

and defines the raw intrinsic reward from forward prediction error:

$$\tilde{r}^{\mathrm{joint}}_t = \frac{1}{2} \left\lVert z_{t+1} - \hat{z}_{t+1} \right\rVert_2^2$$

The provider clips this by the outer curiosity strength `\lambda`, and PPO later scales it again by that same global curiosity strength.


## Modal Curiosity Math

For active modality branches $\mathcal{M}$, each branch gets its own encoder and forward model:

$$z_t^{(m)} = \phi_m(o_t^{(m)})$$

$$\hat{z}_{t+1}^{(m)} = f_m(z_t^{(m)}, u_t)$$

$$\tilde{r}^{(m)}_t = \frac{1}{2} \left\lVert z_{t+1}^{(m)} - \hat{z}_{t+1}^{(m)} \right\rVert_2^2$$

Let the raw branch strengths be $\alpha_m > 0$. The plugin normalizes them internally:

$$w_m = \frac{\alpha_m}{\sum_{j \in \mathcal{M}} \alpha_j}$$

so the branch weights sum to one.

The combined raw modal curiosity reward is then:

$$\tilde{r}^{\mathrm{modal}}_t = \sum_{m \in \mathcal{M}} w_m \tilde{r}^{(m)}_t$$

This is why `1.0` and `0.5` become `2/3` and `1/3`. The branch strengths define composition, while upstream `reward_signals.curiosity.strength` still defines total intrinsic-vs-extrinsic scale.


## Why Fused Curiosity Can Fail

In general, one shared encoder over fused observations does not preserve modality-specific novelty cleanly:

$$\phi(v_t, a_t, x_t) \neq [\phi_v(v_t), \phi_a(a_t), \phi_x(x_t)]$$

So the current fused curiosity reward is not just a clean sum of visual novelty plus auditory novelty. One latent bottleneck, one forward model, and one scalar prediction error means the modalities compete through the same parameter budget.

That can matter if visual structure is a better bootstrap teacher early, while auditory structure is more DV-specific or only pays off later in training.


## Why Separate-But-Simultaneous Curiosity Helps

This design keeps both modalities on, but isolates them before intrinsic reward is formed.

That gives three practical benefits:

- useful visual bootstrap signal is not forced to share a curiosity encoder with auditory structure
- auditory curiosity can stay on without owning the whole latent geometry
- per-branch diagnostics become possible

The weighted-normalized aggregation also keeps overall reward scale stable when additional branches are enabled.


In [ ]:
behaviors:
  2DPoke:
    trainer_type: ppo_modal_curiosity

    hyperparameters:
      modal_curiosity:
        visual:
          strength: 1.0
        auditory:
          strength: 0.5

    reward_signals:
      extrinsic:
        gamma: 0.995
        strength: 1.0
      curiosity:
        gamma: 0.99
        strength: 0.1


## Current Package Choices

- visual branch resolves by project sensor aliases `RatVisualField` and `CameraSensor`
- auditory branch resolves by project sensor aliases `RatAuditoryField` and legacy `Auditory Spectrogram`
- `remaining` means every observation sensor not already claimed by visual or auditory
- branch strengths are normalized internally
- per-branch `gamma` is intentionally deferred because the current design still has one outer curiosity value stream


## Deferred Future Architecture

A later extension may intentionally move to:

- separate modality return targets
- separate modality value heads
- modality-specific `gamma`
- policy-side modality specialization

That is a different reward-learning architecture. `ppo_modal_curiosity` is the smaller first step that makes the intrinsic objective modality-aware without rewriting PPO itself.
